In [15]:
import pandas as pd
import numpy as np
import os

results = pd.read_csv("/results.csv")
drivers = pd.read_csv("/drivers.csv")
races = pd.read_csv("/races.csv")
constructors = pd.read_csv("/constructors.csv")
pit_stops = pd.read_csv("/pit_stops.csv")
df = pd.merge(results, drivers, on="driverId", how="left")
df = pd.merge(df, races, on="raceId", how="left")
df = pd.merge(df, constructors, on="constructorId", how="left")
df.head()
df.shape
df.columns

#Cleaning Data
df = df.rename(columns={
    "name_x": "driver_name",
    "name_y": "constructor_name",
    "nationality_x": "driver_nationality",
    "nationality_y": "constructor_nationality"
})
df = df.drop(columns=["url_x", "url_y", "number_y"])
df = df.drop(columns=[
    "number_x", "time_x", "time_y",
    "fp1_date", "fp1_time", "fp2_date", "fp2_time",
    "fp3_date", "fp3_time",
    "quali_date", "quali_time",
    "sprint_date", "sprint_time",
    "url"
])
df = df.drop(columns=[
    "forename", "surname",
    "statusId",
    "milliseconds",
    "laps"
])
df = df.drop(columns=[
    "position", "positionText",
    "fastestLap", "rank", "fastestLapTime", "fastestLapSpeed",
    "driverRef", "code", "dob"
])
df["is_winner"] = (df["positionOrder"] == 1).astype(int)
df[["driver_name", "constructor_name", "grid", "positionOrder", "is_winner"]].head(10)
df.columns

Index(['resultId', 'raceId', 'driverId', 'constructorId', 'grid',
       'positionOrder', 'points', 'driver_nationality', 'year', 'round',
       'circuitId', 'driver_name', 'date', 'constructorRef',
       'constructor_name', 'constructor_nationality', 'is_winner'],
      dtype='object')

In [16]:
# ── 3. SELECT ONLY THE COLUMNS WE NEED ───────────────────────
#
# WHY? Raw CSVs have many columns we don't need (like Wikipedia
# URLs, timing data in milliseconds, etc.). Keeping only what
# matters makes your dataframe leaner and easier to understand.
#
# Rule of thumb: if you can't explain why a column is useful,
# don't keep it yet.

# results.csv columns we need:
results_cols = [
    'resultId',       # unique ID for each result row
    'raceId',         # links to races.csv
    'driverId',       # links to drivers.csv
    'constructorId',  # links to constructors.csv
    'grid',           # starting grid position
    'position',       # finishing position (NaN if DNF)
    'positionOrder',  # numeric finishing order (always filled)
    'points',         # points scored
    'laps',           # laps completed
    'statusId',       # finish status (finished, retired, etc.)
    'fastestLapTime', # fastest lap time (string format)
    'fastestLapSpeed' # fastest lap speed (kph)
]

# races.csv columns we need:
races_cols = [
    'raceId',    # primary key
    'year',      # season year
    'round',     # race number in the season
    'name',      # Grand Prix name
    'date'       # race date
]

# drivers.csv columns we need:
drivers_cols = [
    'driverId',    # primary key
    'driverRef',   # short name like 'hamilton'
    'forename',    # first name
    'surname',     # last name
    'nationality', # driver nationality
    'dob'          # date of birth (for age calculations later)
]

# constructors.csv columns we need:
constructors_cols = [
    'constructorId',  # primary key
    'name',           # team name like 'Mercedes'
    'nationality'     # team nationality
]

# Apply the selection
results      = results[results_cols]
races        = races[races_cols]
drivers      = drivers[drivers_cols]
constructors = constructors[constructors_cols]

print("Columns selected. Let's verify:")
print(f"results: {results.shape}")
print(f"races: {races.shape}")
print(f"drivers: {drivers.shape}")
print(f"constructors: {constructors.shape}")

Columns selected. Let's verify:
results: (26759, 12)
races: (1125, 5)
drivers: (861, 6)
constructors: (212, 3)


In [17]:
# ── 4. HANDLE MISSING VALUES ──────────────────────────────────
#
# In F1 data, '\\N' means "not applicable" (e.g., a driver who
# retired has no finishing position). Pandas reads this as a
# string, not as NaN. We need to fix that.

# Replace '\\N' with actual NaN across all dataframes
results      = results.replace('\\N', np.nan)
races        = races.replace('\\N', np.nan)
drivers      = drivers.replace('\\N', np.nan)
constructors = constructors.replace('\\N', np.nan)

# Now convert numeric columns that got loaded as strings
# because of the '\\N' values mixed in
numeric_cols = ['grid', 'position', 'positionOrder',
                'points', 'laps', 'fastestLapSpeed']

for col in numeric_cols:
    results[col] = pd.to_numeric(results[col], errors='coerce')
    # errors='coerce' means: if a value can't be converted to
    # a number, just make it NaN instead of crashing

# Convert dates properly
races['date']   = pd.to_datetime(races['date'])
drivers['dob']  = pd.to_datetime(drivers['dob'], errors='coerce')

print("Missing value check after cleaning:")
print(results.isnull().sum())

Missing value check after cleaning:
resultId               0
raceId                 0
driverId               0
constructorId          0
grid                   0
position           10953
positionOrder          0
points                 0
laps                   0
statusId               0
fastestLapTime     18507
fastestLapSpeed    18507
dtype: int64


In [18]:
# ── 5. RENAME COLUMNS TO AVOID CONFLICTS ─────────────────────
#
# When we merge, columns with the same name from different
# tables get renamed with _x and _y suffixes, which is ugly
# and confusing. We rename BEFORE merging to prevent this.
#
# Example: both drivers and constructors have 'nationality'.
# We rename them so after merging we know which is which.

drivers = drivers.rename(columns={
    'nationality': 'driver_nationality',
    'driverRef':   'driver_ref'
})

constructors = constructors.rename(columns={
    'name':        'constructor_name',
    'nationality': 'constructor_nationality'
})

races = races.rename(columns={
    'name': 'race_name'
})

print("Renaming done. Constructors columns:", constructors.columns.tolist())

Renaming done. Constructors columns: ['constructorId', 'constructor_name', 'constructor_nationality']


In [19]:
# ── 6. MERGE INTO MASTER DATAFRAME ───────────────────────────
#
# We use LEFT JOIN (how='left') which means:
# "Keep all rows from the LEFT table (results), and attach
# matching info from the RIGHT table. If no match exists,
# fill with NaN."
#
# This preserves every race result even if some info is missing.
# We merge step by step for clarity.

# Step 1: results + races (adds year, round, race_name, date)
master = pd.merge(
    results,       # left table
    races,         # right table
    on='raceId',   # the column that links them
    how='left'     # keep all results
)
print(f"After merge with races: {master.shape}")

# Step 2: + drivers (adds driver name, nationality, dob)
master = pd.merge(
    master,
    drivers,
    on='driverId',
    how='left'
)
print(f"After merge with drivers: {master.shape}")

# Step 3: + constructors (adds team name, nationality)
master = pd.merge(
    master,
    constructors,
    on='constructorId',
    how='left'
)
print(f"After merge with constructors: {master.shape}")

# Create a full driver name column for convenience
master['driver_name'] = master['forename'] + ' ' + master['surname']

print("\nMaster dataframe preview:")
print(master.head(3))
print(f"\nFinal shape: {master.shape}")
print(f"Columns: {master.columns.tolist()}")

After merge with races: (26759, 16)
After merge with drivers: (26759, 21)
After merge with constructors: (26759, 23)

Master dataframe preview:
   resultId  raceId  driverId  constructorId  grid  position  positionOrder  \
0         1      18         1              1     1       1.0              1   
1         2      18         2              2     5       2.0              2   
2         3      18         3              3     7       3.0              3   

   points  laps  statusId  ...              race_name       date  driver_ref  \
0    10.0    58         1  ...  Australian Grand Prix 2008-03-16    hamilton   
1     8.0    58         1  ...  Australian Grand Prix 2008-03-16    heidfeld   
2     6.0    58         1  ...  Australian Grand Prix 2008-03-16     rosberg   

   forename   surname driver_nationality        dob constructor_name  \
0     Lewis  Hamilton            British 1985-01-07          McLaren   
1      Nick  Heidfeld             German 1977-05-10       BMW Sauber   
2 

In [20]:
# ── 7. CLEAN PIT STOPS SEPARATELY ────────────────────────────
#
# pit_stops.csv needs its own cleaning because:
# - It can have multiple rows per driver per race (each stop)
# - Duration is stored as a string like "22.544"
# - We want to compute aggregates (avg, min stops per race)

pit_cols = ['raceId', 'driverId', 'stop', 'lap', 'duration', 'milliseconds']
pit_stops = pit_stops[pit_cols]
pit_stops = pit_stops.replace('\\N', np.nan)

# Convert duration to float (seconds)
pit_stops['duration'] = pd.to_numeric(pit_stops['duration'], errors='coerce')
pit_stops['milliseconds'] = pd.to_numeric(pit_stops['milliseconds'], errors='coerce')

# Remove clearly bad data: pit stops > 120 seconds are likely
# safety car or mechanical issues, not actual pit work
pit_stops_clean = pit_stops[pit_stops['duration'] < 120].copy()

print(f"Pit stops before cleaning: {pit_stops.shape}")
print(f"Pit stops after cleaning: {pit_stops_clean.shape}")
print(pit_stops_clean.head())

Pit stops before cleaning: (11371, 6)
Pit stops after cleaning: (10854, 6)
   raceId  driverId  stop  lap  duration  milliseconds
0     841       153     1    1    26.898         26898
1     841        30     1    1    25.021         25021
2     841        17     1   11    23.426         23426
3     841         4     1   12    23.251         23251
4     841        13     1   13    23.842         23842


In [23]:
# ── 8. SAVE PROCESSED DATA ───────────────────────────────────
#
# We save to data/processed/ so all future notebooks and the
# Streamlit app can read clean data directly. This way we
# only clean once and reuse everywhere.

processed_path = "../data/processed/"
os.makedirs(processed_path, exist_ok=True)  # create folder if it doesn't exist

master.to_csv(processed_path + "master_df.csv", index=False)
pit_stops_clean.to_csv(processed_path + "pit_stops_clean.csv", index=False)
print(master.info())
print(master.isnull().sum())
print(master.duplicated().sum())
print(master['driver_name'].nunique())
print("✅ Saved master_df.csv")
print("✅ Saved pit_stops_clean.csv")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26759 entries, 0 to 26758
Data columns (total 24 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   resultId                 26759 non-null  int64         
 1   raceId                   26759 non-null  int64         
 2   driverId                 26759 non-null  int64         
 3   constructorId            26759 non-null  int64         
 4   grid                     26759 non-null  int64         
 5   position                 15806 non-null  float64       
 6   positionOrder            26759 non-null  int64         
 7   points                   26759 non-null  float64       
 8   laps                     26759 non-null  int64         
 9   statusId                 26759 non-null  int64         
 10  fastestLapTime           8252 non-null   object        
 11  fastestLapSpeed          8252 non-null   float64       
 12  year                     26759 n